# GATos Team - Final evaluation

**Instructions of usage:** use this notebook alone for the final, held-out test set prediction. The shared
model and automatic Improver use supplied trained artifacts: no neural training, public-row decisions, or human approval.

Place your workbook at `GATos_Team/Input/heldout_test.xlsx` on Drive,
or change FEV001. First-sheet columns: `row_id`, `anon_user_id`, `post`.
Row IDs must be unique text; required cells must be nonempty. Labels are
unnecessary.

Extract the separate artifact bundle to `MyDrive/GATos_Team_Artifacts`.
It supplies `Runs/fresh` and the byte-identical `Input/train.xlsx`.
Do not resave the training workbook. Source ZIPs exclude large weights.
See `docs/FINAL_EVALUATION.md`.

Use **Google Colab Pro+ with Python 3.12.13 / NVIDIA A100 80 GB**. Run all cells with
`CURRENT_RUNTIME = "A"`. 
<p>After A reports PASS, choose **Runtime → Disconnect
and delete runtime**, connect a fresh GPU VM, set B, and run all again.
<p>Repeat with fresh C after B passes. Keep paths and run name unchanged.
<p>Fresh VMs isolate incompatible dependencies; outputs persist on Drive.
C also runs the automatic Improver.

In [ ]:
# FEV001 - Configure held-out input and artifacts.
from pathlib import Path
import re
from google.colab import drive

drive.mount("/content/drive")
CURRENT_RUNTIME = "A"
ROOT = Path("/content/drive/MyDrive/GATos_Team")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/GATos_Team_Artifacts")
ARTIFACTS = ARTIFACT_ROOT / "Runs/fresh"
TRAIN_PATH = ARTIFACT_ROOT / "Input/train.xlsx"
TEST_PATH = ROOT / "Input/heldout_test.xlsx"
RUN_NAME = "heldout_01"
DOWNLOAD_RESULTS = False

if CURRENT_RUNTIME not in {"A", "B", "C"}:
    raise ValueError("Choose CURRENT_RUNTIME A, B, or C.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_NAME):
    raise ValueError("Use a short run name containing letters, digits, underscores, or hyphens.")
EVALUATION = ROOT / "Evaluations" / RUN_NAME
for path in (ROOT / "pyproject.toml", TRAIN_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required file is missing: {path}")
if not (ARTIFACTS / "Models").is_dir():
    raise FileNotFoundError(f"Provide the trained artifact directory: {ARTIFACTS}")
print("Runtime:", CURRENT_RUNTIME)
print("Held-out input:", TEST_PATH)
print("Saved artifacts:", ARTIFACTS)
print("Persistent evaluation:", EVALUATION)


In [ ]:
# FEV002 - Verify Python and GPU allocation.
import subprocess
import sys

advisories = []
if sys.version_info[:3] != (3, 12, 13):
    advisories.append(f"Python {sys.version.split()[0]} instead of 3.12.13")
try:
    gpu = subprocess.check_output([
        "nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits",
    ], text=True).strip()
    name, memory = gpu.splitlines()[0].rsplit(",", 1)
    if "A100" not in name or float(memory) < 75 * 1024:
        advisories.append(f"GPU allocation: {gpu}")
except (FileNotFoundError, subprocess.CalledProcessError, ValueError, IndexError) as error:
    gpu = f"unavailable ({error})"
    advisories.append(f"GPU allocation: {gpu}")
if advisories:
    print("WARNING: You are not running with the advised settings - Google Colab Pro+, Python 3.12.13, A100 80 GB.")
    for advisory in advisories:
        print(" -", advisory)
    print("The frozen preflight may still stop an incompatible run.")
else:
    print("ADVISED RUNTIME: PASS")
print("GPU:", gpu)
print("Python:", sys.version)


In [ ]:
# FEV003 - Install this runtime's dependencies.
import subprocess
import sys

requirements = "requirements-probes.txt" if CURRENT_RUNTIME == "B" else "requirements-train.txt"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", str(ROOT / requirements),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(ROOT),
], check=True)
sys.path.insert(0, str(ROOT / "src"))
print("Installed environment:", requirements)


In [ ]:
# FEV004 - Configure commands and verify the runtime package.
from gatos_release.launch import run_streamed

def run_command(command):
    run_streamed(command, root=ROOT)

def model_phase(phase):
    run_command([
        sys.executable, "-u", "-m", "gatos_release.model",
        "--root", ROOT,
        "--artifacts", ARTIFACTS,
        "--train", TRAIN_PATH,
        "--test", TEST_PATH,
        "--output", EVALUATION,
        phase,
    ])

required_runtime_files = [
    ROOT / "config/pipeline.yaml",
    ROOT / "requirements-train.txt",
    ROOT / "requirements-probes.txt",
]
missing = [str(path) for path in required_runtime_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing runtime files: " + ", ".join(missing))

import gatos.cli
import gatos_release.model
import gatos_release.final_evaluation
import gatos_rule_improver.rules

print("RUNTIME PACKAGE CHECK: PASS")


## A - QLoRA inference

Reuse trained 7B adapters. Replace the VM only after successful completion and
`QLoRA STATE PERSISTED: PASS`.

In [ ]:
# FEV005 - Run QLoRA and persist predictions.
if CURRENT_RUNTIME == "A":
    model_phase("prepare")
    model_phase("qlora-test")
    print("Runtime A: PASS. Continue with the same run name in a fresh Runtime B.")
else:
    print("Skipped: Runtime A.")


## B - Boundary probes

The fixed 72B model scores test posts only, reusing original training probes.
Wait for successful completion and `TEST PROBES PERSISTED: PASS`
before starting a fresh C VM.

In [ ]:
# FEV006 - Generate and persist held-out probes.
if CURRENT_RUNTIME == "B":
    model_phase("probe-test")
    print("Runtime B: PASS. Continue with the same run name in a fresh Runtime C.")
else:
    print("Skipped: Runtime B.")


## C - Model output and automatic refinement

Assemble validated risk, evidence, and factors; apply eligible automatic text rules.
No public-row lookups or review prompts. Uncertain cases retain model predictions.
No held-out score is claimed; see the documented distribution-shift limit.

Drive model loading and integrity checks can be slow. Logs and receipts persist
in the evaluation directory. Never bypass failed guards or substitute a public CSV.

In [ ]:
# FEV007 - Predict, refine, and validate publication.
from datetime import datetime, timezone

FINAL_PATH = None
if CURRENT_RUNTIME == "C":
    model_phase("predict")
    model_phase("status")
    raw_path = EVALUATION / "Output/GATos_model_raw.csv"
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    final_directory = EVALUATION / "FinalEvaluation" / stamp
    run_command([
        sys.executable, "-u", "-m", "gatos_release.final_evaluation",
        "--raw", raw_path,
        "--test", TEST_PATH,
        "--out", final_directory,
    ])
    FINAL_PATH = final_directory / "GATos.csv"
else:
    print("Skipped: Runtime C and final publication.")


In [ ]:
# FEV008 - Verify outputs and optionally download.
import hashlib
import json

if CURRENT_RUNTIME == "C":
    if FINAL_PATH is None:
        raise RuntimeError("Complete FEV007 successfully before collecting predictions.")
    manifest_path = FINAL_PATH.with_suffix(".manifest.json")
    audit_path = FINAL_PATH.parent / "audit.json"
    result = json.loads(manifest_path.read_text(encoding="utf-8"))
    if result["sha256"] != hashlib.sha256(FINAL_PATH.read_bytes()).hexdigest():
        raise RuntimeError("Final CSV differs from its publication record.")
    if result["audit_sha256"] != hashlib.sha256(audit_path.read_bytes()).hexdigest():
        raise RuntimeError("Final audit differs from its publication record.")
    print("FINAL EVALUATION: PASS")
    print("Rows:", result["rows"])
    print("Automatically changed cells:", result["changed_cells"])
    print("Model validation warnings:", result["model_validation_warnings"])
    print("Prediction CSV:", FINAL_PATH)
    print("Audit:", audit_path)
    print("All results:", FINAL_PATH.parent)
    if DOWNLOAD_RESULTS:
        from google.colab import files
        for path in (FINAL_PATH, manifest_path, audit_path):
            files.download(str(path))
else:
    print("Final CSV will be available after Runtime C. No final output is ready yet.")


Output: `Evaluations/<run>/FinalEvaluation/<timestamp>/GATos.csv`.
Keep its manifest, `audit.json`, `semantic_diff.csv`, `review_proposals.json`,
preserved raw CSV, and model-stage logs. Zero automatic changes is valid.
Nothing is submitted or shared. For different input, choose a new run name
and repeat A → B → C. Never run concurrent writers in one evaluation directory.